# Phase 3f — QAOA for Scenario A (LocalSimulator → optional SV1)

This notebook takes the **Scenario A** optimization problem we encoded as a **QUBO** and runs a **QAOA** workflow to produce a quantum (or simulated-quantum) candidate solution.

We will:
1. Load the Scenario A trial universe and the saved QUBO artifacts from Phase 3e.
2. Build a Braket QAOA circuit for the QUBO objective.
3. Run locally first (fast, deterministic smoke test).
4. Optionally run on **SV1** (statevector) if Braket results storage is configured correctly.
5. Persist results artifacts to `data/results/` for GitHub tracking and comparisons.


In [3]:
# ============================================================
# Cell 1 — Imports + Load Scenario A + Load QUBO artifacts (your JSON schema)
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

SCENARIO_A_TRIALS = Path("data/scenarios/scenario_A_trials.csv")
QUBO_JSON         = Path("data/qubo/scenario_A_qubo.json")

RESULTS_DIR = Path("data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Load QUBO JSON first (so we can use the canonical nct_id order) ---
with open(QUBO_JSON, "r") as f:
    qubo_payload = json.load(f)

print("[Cell 1] scenario_A_qubo.json keys:", list(qubo_payload.keys()))

# Pull Q + metadata from your schema
Q = np.array(qubo_payload["Q_dense"], dtype=float)
offset = float(qubo_payload.get("offset", 0.0))  # may be absent; default 0.0

n_vars = int(qubo_payload["n_variables"])
nct_id_order = list(qubo_payload["nct_ids"])

# Sanity checks
assert Q.shape == (n_vars, n_vars), f"Expected Q shape {(n_vars, n_vars)}, got {Q.shape}"
assert len(nct_id_order) == n_vars, "nct_ids length must match n_variables"

# --- Load Scenario A trials and align to QUBO variable order ---
dfA_trials = pd.read_csv(SCENARIO_A_TRIALS)
if "nct_id" not in dfA_trials.columns:
    raise ValueError("Expected 'nct_id' column in scenario_A_trials.csv")

# Align rows to the QUBO's nct_ids ordering (THIS is the source of truth)
dfA_trials = (
    dfA_trials.set_index("nct_id")
    .reindex(nct_id_order)
    .reset_index()
)

missing = dfA_trials["nct_id"].isna().sum()
if missing:
    raise ValueError(
        f"{missing} nct_ids from the QUBO were not found in scenario_A_trials.csv. "
        "This indicates a mismatch between the QUBO export order and the trials file."
    )

print("[Cell 1] Loaded Scenario A trials (aligned):", dfA_trials.shape)
print("[Cell 1] Loaded QUBO Q shape:", Q.shape, "| offset:", offset)
print("[Cell 1] QUBO params:",
      "K=", qubo_payload.get("portfolio_k"),
      "lambda_cost=", qubo_payload.get("lambda_cost"),
      "lambda_safety=", qubo_payload.get("lambda_safety"),
      "penalty_k=", qubo_payload.get("penalty_k"))


[Cell 1] scenario_A_qubo.json keys: ['scenario', 'created_utc', 'n_variables', 'portfolio_k', 'lambda_cost', 'lambda_safety', 'penalty_k', 'nct_ids', 'Q_dense']
[Cell 1] Loaded Scenario A trials (aligned): (18, 13)
[Cell 1] Loaded QUBO Q shape: (18, 18) | offset: 0.0
[Cell 1] QUBO params: K= 6 lambda_cost= 1.0 lambda_safety= 1.0 penalty_k= 50.0


### What Cell 1 Just Did

This cell loaded the Scenario A trial table and the exported QUBO artifact (`data/qubo/scenario_A_qubo.json`). It pulled the dense QUBO matrix from the `Q_dense` field and, critically, aligned the Scenario A trials to the exact `nct_ids` ordering stored in the QUBO JSON.

That alignment is required because every bit in a bitstring (or every qubit in a circuit state) corresponds to one variable in the QUBO, and that variable order is defined by `nct_ids`. With this cell complete, we now have:
- `Q` (the QUBO matrix) and `offset`
- `n_vars` and the canonical `nct_id_order`
- `dfA_trials` indexed in the same order as the QUBO variables


In [5]:
# ============================================================
# Cell 2 — QUBO utilities (energy, decoding, sampling)
# ============================================================
#
# Purpose:
#   - Provide small, reusable helpers to:
#       * Compute QUBO energy for a bitstring (minimization)
#       * Convert between bitstrings and numpy vectors
#       * Enforce/score a cardinality constraint (select exactly K)
#       * Decode selected trial rows from a solution vector
# ============================================================

import math
import numpy as np

# Pull K if present (used for feasibility checks / decoding)
K = int(qubo_payload.get("portfolio_k", 0)) if "qubo_payload" in globals() else 0
print("[Cell 2] portfolio_k (K):", K)

def bitstring_to_x(bitstring: str) -> np.ndarray:
    """
    "0101" -> np.array([0,1,0,1], dtype=int)
    Assumes qubit0-first ordering (leftmost char = x0).
    """
    bitstring = bitstring.strip()
    return np.fromiter((1 if ch == "1" else 0 for ch in bitstring), dtype=np.int8)

def x_to_bitstring(x: np.ndarray) -> str:
    """np.array([0,1,0,1]) -> "0101" (qubit0-first)."""
    return "".join("1" if v else "0" for v in x.astype(int).tolist())

def qubo_energy(Q: np.ndarray, x: np.ndarray, offset: float = 0.0) -> float:
    """
    Compute E(x) = x^T Q x + offset for x in {0,1}^n.
    Lower energy is better.
    """
    x = x.astype(float)
    return float(x @ Q @ x + offset)

def cardinality(x: np.ndarray) -> int:
    """Number of selected items (ones)."""
    return int(x.sum())

def is_feasible_k(x: np.ndarray, K: int) -> bool:
    """Feasible if exactly K selections (common portfolio-style constraint)."""
    return cardinality(x) == int(K)

def decode_selected_trials(df_trials: pd.DataFrame, x: np.ndarray) -> pd.DataFrame:
    """
    Return subset of df_trials where x_i == 1.
    Assumes df_trials is aligned to QUBO variable order.
    """
    idx = np.where(x.astype(int) == 1)[0]
    selected = df_trials.iloc[idx].copy()
    selected["qubo_var_index"] = idx
    return selected

def sample_bitstrings_from_statevector(sv: np.ndarray, n: int, shots: int, rng=None):
    """
    Sample measurement outcomes from a statevector.
    Returns a list of bitstrings in qubit0-first ordering.
    """
    if rng is None:
        rng = np.random.default_rng()

    probs = np.abs(sv) ** 2
    probs = probs / probs.sum()

    # Sample basis state indices
    idxs = rng.choice(len(probs), size=shots, p=probs)

    # Convert basis index -> bitstring.
    # Numpy binary representation gives MSB-first, so we format to n bits,
    # then interpret leftmost as qubit0 (qubit0-first).
    # If your earlier notebooks used the opposite convention, flip here.
    bitstrings = [format(i, f"0{n}b") for i in idxs]
    return bitstrings

[Cell 2] portfolio_k (K): 6


### What Cell 2 Just Did

This cell introduced small helper functions that we will reuse throughout the notebook:
- Conversions between bitstrings and binary vectors (`x`)
- A consistent QUBO energy evaluator (`xᵀQx + offset`) for minimization
- Cardinality/feasibility checks for “select exactly K trials”
- A decoder that turns a solution vector into a selected-trials DataFrame
- A statevector sampling helper for estimating good bitstrings from quantum outputs


In [6]:
# ============================================================
# Cell 3 — Braket configuration + runners (Local + SV1)
# ============================================================
#
# Purpose:
#   - Configure AWS/Braket execution for both LocalSimulator and SV1.
#   - IMPORTANT: Braket task results must be written to a bucket that
#     starts with "amazon-braket-". We therefore use AwsSession.default_bucket()
#     instead of your project bucket.
# ============================================================

import os
import boto3
from braket.aws import AwsSession, AwsDevice
from braket.devices import LocalSimulator
from braket.circuits import Circuit

AWS_REGION = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-west-2"
boto_sess = boto3.Session(region_name=AWS_REGION)
aws_sess = AwsSession(boto_session=boto_sess)

# Braket-managed default bucket (guaranteed to have amazon-braket- prefix)
BRAKET_RESULTS_BUCKET = aws_sess.default_bucket()
BRAKET_RESULTS_PREFIX = "quantum-clinical-trial-optimization/phase3"

print("[Cell 3] AWS_REGION:", AWS_REGION)
print("[Cell 3] Braket results bucket:", BRAKET_RESULTS_BUCKET)
print("[Cell 3] Braket results prefix:", BRAKET_RESULTS_PREFIX)

# Local simulators
local_sv = LocalSimulator("braket_sv")  # statevector simulator

def run_statevector_local(circuit: Circuit):
    """
    Run a circuit on LocalSimulator to obtain statevector.
    Requires circuit.state_vector() result type to be present.
    """
    task = local_sv.run(circuit, shots=0)
    return task.result()

def run_statevector_sv1(circuit: Circuit):
    """
    Run a circuit on Amazon SV1 (managed statevector simulator).
    Results go to the Braket default bucket (amazon-braket-*).
    """
    device = AwsDevice("arn:aws:braket:::device/quantum-simulator/amazon/sv1", aws_session=aws_sess)
    task = device.run(
        circuit,
        s3_destination_folder=(BRAKET_RESULTS_BUCKET, BRAKET_RESULTS_PREFIX),
        shots=0,
    )
    return task.result()

def statevector_from_task_result(res):
    """
    Extract the statevector from a Braket task result.
    Works when the circuit includes circuit.state_vector().
    """
    if hasattr(res, "values") and res.values:
        return np.asarray(res.values[0], dtype=complex)

    # Fallbacks (rare)
    if hasattr(res, "result_types") and res.result_types:
        # Some older SDK shapes expose values differently
        try:
            return np.asarray(res.result_types[0].value, dtype=complex)
        except Exception:
            pass

    raise ValueError("Could not extract a statevector from the Braket result. "
                     "Ensure the circuit has circuit.state_vector() added.")


[Cell 3] AWS_REGION: us-west-2
[Cell 3] Braket results bucket: amazon-braket-us-west-2-581610642254
[Cell 3] Braket results prefix: quantum-clinical-trial-optimization/phase3


### What Cell 3 Just Did

This cell set up the execution layer for both local and AWS runs. It creates an `AwsSession` in the target region and uses `aws_sess.default_bucket()` to pick a Braket-managed results bucket. This avoids the `ValidationException` you saw earlier where Braket rejected a non–`amazon-braket-*` bucket.

It also defines small runner functions:
- `run_statevector_local()` for fast local statevector simulation
- `run_statevector_sv1()` for managed SV1 statevector simulation
- `statevector_from_task_result()` to consistently extract the statevector from results


In [7]:
# ============================================================
# Cell 4 — QUBO -> Ising mapping + QAOA circuit builder
# ============================================================
#
# Purpose:
#   - Convert QUBO cost C(x)=x^T Q x (+offset) into an Ising form
#       H(z) = const + sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
#     using x_i = (1 - Z_i)/2.
#   - Build a p-layer QAOA circuit for that Ising Hamiltonian.
#
# Notes:
#   - Global phase (the constant term) does not affect measurement probabilities,
#     so we compute it for reporting but do not need to implement it in the circuit.
# ============================================================

def qubo_to_ising(Q: np.ndarray, offset: float = 0.0):
    """
    Convert QUBO to Ising coefficients.

    Returns:
      h: (n,) array
      J: list of (i,j, Jij) with i<j
      const: float
    """
    Qsym = 0.5 * (Q + Q.T)
    n = Qsym.shape[0]

    h = np.zeros(n, dtype=float)
    const = float(offset)
    J_terms = []

    # Diagonal: b_i * x_i -> const += b_i/2; h_i += -b_i/2
    for i in range(n):
        b = float(Qsym[i, i])
        const += 0.5 * b
        h[i] += -0.5 * b

    # Off-diagonal: energy uses x^T Q x, so pair coefficient is a_ij = 2*Qsym[i,j]
    for i in range(n):
        for j in range(i + 1, n):
            q = float(Qsym[i, j])
            if q == 0.0:
                continue
            a = 2.0 * q  # coefficient on x_i x_j
            const += 0.25 * a
            h[i] += -0.25 * a
            h[j] += -0.25 * a
            Jij = 0.25 * a
            J_terms.append((i, j, Jij))

    return h, J_terms, const

h_ising, J_ising, const_ising = qubo_to_ising(Q, offset=offset)

print("[Cell 4] Ising terms:", "n=", len(h_ising), "| J_terms=", len(J_ising))
print("[Cell 4] Ising constant (global phase, not implemented):", const_ising)

def apply_zz_interaction(circ: Circuit, i: int, j: int, angle: float):
    """
    Apply exp(-i * angle/2 * Z_i Z_j) up to a global phase
    via CNOT-RZ-CNOT pattern.

    We implement exp(-i * gamma * Jij * Z_i Z_j) as:
      CNOT(i,j); RZ(2*gamma*Jij) on j; CNOT(i,j)
    """
    circ.cnot(i, j)
    circ.rz(j, angle)
    circ.cnot(i, j)

def build_qaoa_circuit(p: int, gammas: np.ndarray, betas: np.ndarray) -> Circuit:
    """
    Build a p-layer QAOA circuit with:
      - Cost unitary from (h_ising, J_ising)
      - Mixer unitary as RX on all qubits
      - Adds state_vector result type.

    Expects gammas, betas shape == (p,)
    """
    n = len(h_ising)
    circ = Circuit()

    # Start in uniform superposition
    for q in range(n):
        circ.h(q)

    # Alternating operators
    for layer in range(p):
        gamma = float(gammas[layer])
        beta = float(betas[layer])

        # Cost: exp(-i gamma * sum h_i Z_i) -> RZ(2*gamma*h_i)
        for i in range(n):
            hi = float(h_ising[i])
            if hi != 0.0:
                circ.rz(i, 2.0 * gamma * hi)

        # Cost: exp(-i gamma * sum J_ij Z_i Z_j) -> CNOT-RZ-CNOT with angle (2*gamma*Jij)
        for (i, j, Jij) in J_ising:
            if Jij != 0.0:
                apply_zz_interaction(circ, i, j, 2.0 * gamma * float(Jij))

        # Mixer: exp(-i beta * sum X_i) -> RX(2*beta)
        for q in range(n):
            circ.rx(q, 2.0 * beta)

    # Request statevector in results
    circ.state_vector()
    return circ

[Cell 4] Ising terms: n= 18 | J_terms= 153
[Cell 4] Ising constant (global phase, not implemented): 2693.647058823529


### What Cell 4 Just Did

This cell converted the QUBO in `Q` into an equivalent Ising Hamiltonian using the standard mapping `x = (1 − Z)/2`. That produces:
- `h_ising` (single-qubit Z weights)
- `J_ising` (pairwise Z⊗Z couplings)
- `const_ising` (a constant global phase term)

It also defined a QAOA circuit builder. For each QAOA layer, we apply:
- The cost unitary using `RZ` gates (for h terms) and a CNOT–RZ–CNOT construction (for J terms)
- A mixer unitary using `RX` gates

Finally, we add `state_vector()` so LocalSimulator/SV1 return a full statevector for analysis.


In [8]:
# ============================================================
# Cell 5 — Local parameter sweep to select best QAOA parameters
# ============================================================
#
# Purpose:
#   - Run a small grid search over (gamma, beta) on LocalSimulator.
#   - Score each candidate using Monte Carlo sampling from the statevector:
#       expected_qubo_cost ≈ mean(qubo_energy(bitstring_samples))
#
# Why sampling:
#   - Exact expectation over all 2^n states can be expensive as n grows.
#   - Sampling is fast and sufficient to pick a good parameter setting.
# ============================================================

import itertools
import time

P = 1  # start with p=1; scale later
GAMMA_GRID = np.linspace(0.1, 2.0, 8)   # coarse grid
BETA_GRID  = np.linspace(0.1, 2.0, 8)

SWEEP_SAMPLES = 3000
rng = np.random.default_rng(42)

def estimate_expected_cost_from_statevector(sv: np.ndarray, Q: np.ndarray, offset: float, samples: int) -> float:
    """Estimate expected QUBO energy by sampling bitstrings from a statevector."""
    n = int(round(math.log2(len(sv))))
    bss = sample_bitstrings_from_statevector(sv, n=n, shots=samples, rng=rng)
    costs = []
    for bs in bss:
        x = bitstring_to_x(bs)
        costs.append(qubo_energy(Q, x, offset=offset))
    return float(np.mean(costs))

best = {"exp_cost": float("inf"), "gammas": None, "betas": None}

t0 = time.time()
candidates = list(itertools.product(GAMMA_GRID, BETA_GRID))
print(f"[Cell 5] Starting local sweep over {len(candidates)} candidates (p={P})...")

for gamma, beta in candidates:
    gammas = np.array([gamma], dtype=float)
    betas  = np.array([beta], dtype=float)

    circ = build_qaoa_circuit(p=P, gammas=gammas, betas=betas)
    res = run_statevector_local(circ)
    sv = statevector_from_task_result(res)

    exp_cost = estimate_expected_cost_from_statevector(sv, Q, offset, samples=SWEEP_SAMPLES)

    if exp_cost < best["exp_cost"]:
        best.update({"exp_cost": exp_cost, "gammas": gammas, "betas": betas})
        print(f"[Cell 5] New best: exp_cost={exp_cost:.6f}  gamma={gamma:.3f}  beta={beta:.3f}")

t1 = time.time()
print(f"[Cell 5] Sweep complete in {t1 - t0:.1f}s")
print("[Cell 5] Best params:", best)


[Cell 5] Starting local sweep over 64 candidates (p=1)...
[Cell 5] New best: exp_cost=2776.661529  gamma=0.100  beta=0.100
[Cell 5] New best: exp_cost=2686.157882  gamma=0.100  beta=0.371
[Cell 5] New best: exp_cost=2590.392275  gamma=0.100  beta=2.000
[Cell 5] New best: exp_cost=2576.286667  gamma=0.643  beta=1.729
[Cell 5] New best: exp_cost=2483.591490  gamma=0.914  beta=0.371
[Cell 5] New best: exp_cost=2346.235294  gamma=0.914  beta=0.643
[Cell 5] New best: exp_cost=2311.109059  gamma=0.914  beta=0.914
[Cell 5] New best: exp_cost=1960.078980  gamma=1.729  beta=0.371
[Cell 5] New best: exp_cost=1582.833922  gamma=1.729  beta=0.643
[Cell 5] Sweep complete in 15.8s
[Cell 5] Best params: {'exp_cost': 1582.8339215686271, 'gammas': array([1.72857143]), 'betas': array([0.64285714])}


### What Cell 5 Just Did

This cell performed a coarse parameter search for a 1-layer QAOA circuit using the local statevector simulator. For each (γ, β) pair in the grid, it:
1. Built a QAOA circuit using the Ising coefficients derived from the QUBO
2. Simulated the circuit locally to obtain a statevector
3. Estimated the expected QUBO cost by sampling bitstrings from that statevector
4. Kept the parameter setting that achieved the lowest estimated expected cost

The result is a “best so far” set of QAOA parameters we can now validate and (optionally) run on SV1.


In [9]:
# ============================================================
# Cell 6 — Final run (SV1 optional), decode selection, write artifacts
# ============================================================
#
# Purpose:
#   - Run the best circuit either locally or on SV1.
#   - Sample bitstrings and pick the best feasible (exactly K ones) solution.
#   - Save:
#       * best bitstring
#       * selected trials CSV
#       * summary CSV
# ============================================================

FINAL_RUN_ON_SV1 = True   # set False if you want local only
FINAL_SAMPLES = 20000     # more samples for a better best-bitstring search
TAG = "03e_scenario_A"    # used for artifact filenames

# Build best circuit (p=1 here; expand later if desired)
circ_best = build_qaoa_circuit(p=P, gammas=best["gammas"], betas=best["betas"])

# Run circuit
use_sv1_final = FINAL_RUN_ON_SV1
print(f"[Cell 6] Final run on {'SV1' if use_sv1_final else 'LocalSimulator'}...")

try:
    res_final = run_statevector_sv1(circ_best) if use_sv1_final else run_statevector_local(circ_best)
except Exception as e:
    print("[Cell 6] SV1 run failed; falling back to LocalSimulator. Error:", repr(e))
    res_final = run_statevector_local(circ_best)

sv_final = statevector_from_task_result(res_final)

# Estimate expected cost from the final statevector
exp_cost_final = estimate_expected_cost_from_statevector(sv_final, Q, offset, samples=SWEEP_SAMPLES)
print(f"[Cell 6] Estimated expected cost (sampled): {exp_cost_final:.6f}")

# Sample bitstrings and choose the best feasible
n = int(round(math.log2(len(sv_final))))
bitstrings = sample_bitstrings_from_statevector(sv_final, n=n, shots=FINAL_SAMPLES, rng=rng)

best_bs = None
best_cost = float("inf")
best_x = None

for bs in bitstrings:
    x = bitstring_to_x(bs)

    # If K is set, enforce it; if K is missing/0, accept any
    if K and not is_feasible_k(x, K):
        continue

    cost = qubo_energy(Q, x, offset=offset)
    if cost < best_cost:
        best_cost = cost
        best_bs = bs
        best_x = x

if best_x is None:
    raise RuntimeError(
        "Could not find a feasible bitstring in samples. "
        "Increase FINAL_SAMPLES or temporarily relax feasibility."
    )

print("[Cell 6] Best feasible sampled bitstring:", best_bs)
print("[Cell 6] Best feasible sampled cost:", best_cost)
print("[Cell 6] Cardinality:", cardinality(best_x))

# Decode selected trials
df_selected = decode_selected_trials(dfA_trials, best_x)

# Optional: compute simple decompositions if columns exist
def _first_existing(cols):
    for c in cols:
        if c in dfA_trials.columns:
            return c
    return None

benefit_col = _first_existing(["benefit_score", "benefit", "benefit_total", "benefit_norm"])
cost_col    = _first_existing(["estimated_trial_cost", "trial_cost", "cost"])
safety_col  = _first_existing(["safety_score", "sponsor_safety_score"])

summary = {
    "tag": TAG,
    "n_variables": int(n_vars),
    "portfolio_k": int(K),
    "final_run_device": "SV1" if use_sv1_final else "LocalSimulator",
    "p_layers": int(P),
    "gamma": float(best["gammas"][0]),
    "beta": float(best["betas"][0]),
    "estimated_expected_cost_sampled": float(exp_cost_final),
    "best_sampled_cost": float(best_cost),
    "best_sampled_cardinality": int(cardinality(best_x)),
}

# Add decomposition if available
if benefit_col:
    summary["selected_benefit_sum"] = float(df_selected[benefit_col].fillna(0).sum())
if cost_col:
    summary["selected_cost_sum"] = float(df_selected[cost_col].fillna(0).sum())
if safety_col:
    summary["selected_safety_sum"] = float(df_selected[safety_col].fillna(0).sum())

# Write artifacts
bitstring_path = RESULTS_DIR / f"{TAG}_best_bitstring_qubit0_first.txt"
selected_path  = RESULTS_DIR / f"{TAG}_selected_trials.csv"
summary_path   = RESULTS_DIR / f"{TAG}_summary.csv"

bitstring_path.write_text(best_bs + "\n")
df_selected.to_csv(selected_path, index=False)
pd.DataFrame([summary]).to_csv(summary_path, index=False)

print("[Cell 6] Wrote:", bitstring_path)
print("[Cell 6] Wrote:", selected_path)
print("[Cell 6] Wrote:", summary_path)

df_selected.head()


[Cell 6] Final run on SV1...
[Cell 6] SV1 run failed; falling back to LocalSimulator. Error: ValidationException('An error occurred (ValidationException) when calling the CreateQuantumTask operation: [line 516] result type state_vector is not supported on the requested device')
[Cell 6] Estimated expected cost (sampled): 1673.799529
[Cell 6] Best feasible sampled bitstring: 010001001100011000
[Cell 6] Best feasible sampled cost: -304.2352941176473
[Cell 6] Cardinality: 6
[Cell 6] Wrote: data/results/03e_scenario_A_best_bitstring_qubit0_first.txt
[Cell 6] Wrote: data/results/03e_scenario_A_selected_trials.csv
[Cell 6] Wrote: data/results/03e_scenario_A_summary.csv


,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score,qubo_var_index
1,NCT05823714,Venetoclax+Azacytidine+Modified BUCY Condition...,Recruiting,Phase 2,['Allogeneic Hematopoietic Stem Cell Transplan...,['VEN+AZA+Modified BUCY'],['China'],The First Affiliated Hospital of Soochow Unive...,The First Affiliated Hospital of Soochow Unive...,Global / Multi-Region,2.0,1.0,0.6,1
5,NCT05821556,Valproic Acid/Simvastatin Plus Gemcitabine/Nab...,Recruiting,Phase 2,['Adenocarcinoma of the Pancreas'],['Valproic acid' 'Simvastatin 20mg' 'Gemcitabi...,['Italy' 'Spain'],"National Cancer Institute, Naples","National Cancer Institute, Naples",Global / Multi-Region,2.0,1.0,0.6,5
8,NCT05820906,Cadonilimab Plus Regorafenib and Gem-Cis Chemo...,Recruiting,Phase 2,['Advanced Biliary Tract Cancer'],['Cadonilimab+Regorafenib+GC'],['China'],Tianjin Medical University Cancer Institute an...,Tianjin Medical University Cancer Institute an...,Global / Multi-Region,2.0,1.0,0.6,8
9,NCT00999804,Extension Study of Lapatinib Plus Herceptin Wi...,"Active, not recruiting",Phase 2,['Breast Cancer'],['Lapatinib' 'Letrozole' 'Trastuzumab'],['United States'],Baylor Breast Care Center,Baylor Breast Care Center,Global / Multi-Region,2.0,1.0,0.6,9
13,NCT05816746,Decitabine and Anti-PD-1 in R/R DLBCL,Recruiting,Phase 2,['Diffuse Large B Cell Lymphoma' 'Relapse/Recu...,['Low-Dose Decitabine plus anti-PD-1'],['China'],Chinese PLA General Hospital,Chinese PLA General Hospital,Global / Multi-Region,2.0,1.0,0.6,13


### What Cell 6 Just Did

This cell executed the “best” QAOA circuit and turned the quantum output back into usable trial selections:

1. It ran the circuit either on SV1 (preferred) or fell back to LocalSimulator if SV1 failed.
2. It estimated expected QUBO performance by sampling from the final statevector.
3. It sampled many candidate bitstrings from the statevector and selected the best feasible solution (exactly K selected trials, when K is defined).
4. It decoded that bitstring into a selected-trials table using the QUBO’s canonical variable ordering.
5. It wrote reproducible artifacts (bitstring, selected trials CSV, and a one-row summary CSV) to `data/results/`.

With these outputs saved, we now have a concrete Scenario A selection result produced by a QAOA-based workflow, ready for comparison against classical baselines and for iteration on p-layers, parameter search, and objective weighting.


## Notebook Summary — Scenario A QAOA Run (SV1-ready)

In this notebook we loaded the exported Scenario A QUBO (`scenario_A_qubo.json`) and aligned the Scenario A trial table to the QUBO’s canonical `nct_ids` ordering so that bitstrings decode correctly.

We then:
- Converted the QUBO to an Ising form using `x = (1 − Z)/2`
- Built a parameterized QAOA circuit that implements the cost unitary (Z and ZZ terms) and a standard X-mixer
- Performed a small local parameter sweep to pick a good (γ, β)
- Ran the best circuit (SV1 optional) and sampled the resulting statevector to find a best feasible bitstring
- Persisted the run artifacts to `data/results/`:
  - `{TAG}_best_bitstring_qubit0_first.txt`
  - `{TAG}_selected_trials.csv`
  - `{TAG}_summary.csv`

Next steps:
1. Increase QAOA depth `p` (e.g., p=2) and expand the parameter search strategy (random search or multi-start).
2. Add a classical baseline comparison for the same Scenario A objective (greedy / simulated annealing) and store side-by-side results.
3. Tighten feasibility handling (exact-K by construction) and standardize a single bit-order convention across notebooks.
